# 01 — Скачивание данных (Sentinel-2, Landsat, RGI)

**Требует:** аутентификации Google Earth Engine (`earthengine authenticate` в терминале)
и доступа к интернету. Этот ноутбук НЕ выполняется в облачной песочнице —
запускайте его локально (Anaconda + `conda activate glaciers`).

Результат: GeoTIFF-композиты в `data/raw/sentinel2/` и `data/raw/landsat/`
(после скачивания вручную из Google Drive, папка `GlacierKZ`),
а также контуры RGI в `data/rgi/`.


In [ ]:
import sys
sys.path.append('..')

import ee
import geemap
import os

from src import config, data_loader

ee.Initialize()
print(ee.String('Earth Engine работает!').getInfo())


## Область исследования

Используем `config.STUDY_AREA_BBOX` — Заилийский Алатау, бассейн рек
Малая/Большая Алматинка (ледники Туюксу, Богдановича и др.).


In [ ]:
STUDY_AREA = ee.Geometry.BBox(*config.STUDY_AREA_BBOX)

m = geemap.Map()
m.centerObject(STUDY_AREA, 10)
m.addLayer(STUDY_AREA, {}, 'Study area')
m


## Sentinel-2 (2015–2024)

In [ ]:
os.makedirs(config.DATA_RAW_SENTINEL2, exist_ok=True)

for year in config.YEARS_SENTINEL2:
    print(f"\n--- Обрабатываем {year} ---")
    img = data_loader.get_sentinel2(year, STUDY_AREA)
    if img is not None:
        print(f"  Каналы: {img.bandNames().getInfo()}")
        data_loader.export_year_to_drive(img, year, STUDY_AREA, prefix='sentinel2')
    else:
        print(f"  Нет данных за {year}, пропускаем")

print("\nВсе задачи запущены! Проверь прогресс на code.earthengine.google.com/tasks")
print("После завершения — скачай файлы из Google Drive (папка GlacierKZ) в data/raw/sentinel2/")
print("Индексы NDSI/NDWI/BSI/EVI будут добавлены локально в data_loader.load_image().")


## Landsat (2000–2013)

In [ ]:
os.makedirs(config.DATA_RAW_LANDSAT, exist_ok=True)

for year in config.YEARS_LANDSAT:
    print(f"\n--- Обрабатываем {year} (Landsat) ---")
    img = data_loader.get_landsat(year, STUDY_AREA)
    if img is not None:
        img = data_loader.add_indices(img)
        data_loader.export_year_to_drive(img, year, STUDY_AREA, prefix='landsat')
    else:
        print(f"  Нет данных за {year}, пропускаем")

print("\nЗадачи Landsat запущены. Скачай результаты в data/raw/landsat/")


## RGI 7.0 — контуры ледников

Скачай регион 13 (Central Asia) с https://www.glims.org/RGI/rgi70_dl.html
(или через `ee.FeatureCollection('GLIMS/current')`), сохрани shapefile в `data/rgi/`.


In [ ]:
# Вариант: контуры ледников прямо из Earth Engine (GLIMS)
rgi_fc = ee.FeatureCollection('GLIMS/current')
kz_glaciers = rgi_fc.filterBounds(STUDY_AREA)
print('Ледников в области исследования:', kz_glaciers.size().getInfo())

# Экспорт в Drive как shapefile/GeoJSON для дальнейшей работы в geopandas/QGIS
task = ee.batch.Export.table.toDrive(
    collection=kz_glaciers,
    description='rgi_study_area',
    folder='GlacierKZ',
    fileNamePrefix='rgi_study_area',
    fileFormat='SHP',
)
task.start()
print('Экспорт контуров RGI запущен -> data/rgi/rgi_study_area.shp')


## Чек-лист после выполнения

- [ ] Sentinel-2 за минимум 5 лет (2015–2024) скачаны в `data/raw/sentinel2/`
- [ ] Landsat за 2000–2013 скачаны в `data/raw/landsat/`
- [ ] RGI контуры в `data/rgi/`
- [ ] Все файлы открываются в QGIS без ошибок (Layer -> Add Raster/Vector Layer)
